# PHẦN VI — KOLLA-ANSIBLE DEPLOY OPENSTACK

## Mục tiêu
Triển khai OpenStack bằng Kolla-Ansible trên cụm 3 Controller VM + 3 Compute bare-metal, tích hợp Ceph backend đã bootstrap ở PHẦN V.

## Thực hiện trên Controller VM1
```bash
ssh ubuntu@10.0.0.11
# hoặc
ssh ubuntu@192.168.150.21
```

## Topology

| Role | IP | Ghi chú |
|---|---|---|
| Controller 1 | `10.0.0.11` | deployment node chính |
| Controller 2 | `10.0.0.12` | control plane |
| Controller 3 | `10.0.0.13` | control plane |
| Compute 1 | `10.0.0.21` | server vật lý A |
| Compute 2 | `10.0.0.22` | server vật lý B |
| Compute 3 | `10.0.0.23` | server vật lý C |

## Điều kiện trước khi deploy
- Controller VM1/2/3 đã hoạt động.
- Compute bare-metal `10.0.0.21/22/23` SSH được bằng user `ubuntu`.
- Ceph đã có pool `images`, `volumes`, `vms`, `backups`.
- Ceph config/keyring đã copy sang `/tmp/` trên Controller VM1.
- Venv Kolla đã có tại `/opt/kolla-venv/`.


## BƯỚC 24 — Chuẩn bị cấu hình Kolla-Ansible

Mục tiêu:
1. Activate venv.
2. Copy cấu hình mẫu vào `/etc/kolla`.
3. Chuẩn bị Ceph config/keyring cho Glance, Cinder, Nova.


In [ ]:
# ============================================================
# BƯỚC 24.1 — SSH vào Controller VM1 và kiểm tra môi trường
# ============================================================

hostname
hostname -f
ip addr
ip route
ping -c 3 8.8.8.8
ping -c 3 google.com


In [ ]:
# ============================================================
# BƯỚC 24.2 — Kích hoạt Kolla-Ansible venv
# ============================================================

source /opt/kolla-venv/bin/activate

which python
python --version

which kolla-ansible
kolla-ansible --version

# Kỳ vọng:
# kolla-ansible 2025.1.x


In [ ]:
# ============================================================
# BƯỚC 24.3 — Tạo /etc/kolla và copy template config
# ============================================================

sudo mkdir -p /etc/kolla
sudo chown -R ubuntu:ubuntu /etc/kolla

cp -r /opt/kolla-venv/share/kolla-ansible/etc_examples/kolla/* /etc/kolla/

ls -lah /etc/kolla
ls -lah /etc/kolla/globals.yml /etc/kolla/passwords.yml


In [ ]:
# ============================================================
# BƯỚC 24.4 — Tạo thư mục config Ceph cho Kolla
# ============================================================

sudo mkdir -p /etc/ceph

sudo mkdir -p /etc/kolla/config/cinder/cinder-volume
sudo mkdir -p /etc/kolla/config/nova
sudo mkdir -p /etc/kolla/config/glance

sudo chown -R ubuntu:ubuntu /etc/kolla/config


In [ ]:
# ============================================================
# BƯỚC 24.5 — Copy Ceph config/keyring từ /tmp vào /etc/ceph
# ============================================================

ls -lh /tmp/ceph.conf
ls -lh /tmp/ceph.client.*.keyring

sudo cp /tmp/ceph.conf /etc/ceph/
sudo cp /tmp/ceph.client.*.keyring /etc/ceph/

sudo chmod 644 /etc/ceph/ceph.conf
sudo chmod 600 /etc/ceph/ceph.client.*.keyring

sudo ls -lh /etc/ceph/


sudo chmod 644 /etc/kolla/config/*/ceph.conf
sudo chmod 600 /etc/kolla/config/*/*.keyring
sudo chmod 600 /etc/kolla/config/cinder/cinder-volume/*.keyring

In [ ]:
# ============================================================
# BƯỚC 24.6 — Copy Ceph keyring vào Kolla config
# ============================================================

# Glance
sudo cp /etc/ceph/ceph.client.glance.keyring /etc/kolla/config/glance/
sudo cp /etc/ceph/ceph.conf /etc/kolla/config/glance/

# Cinder
sudo cp /etc/ceph/ceph.client.cinder.keyring /etc/kolla/config/cinder/cinder-volume/
sudo cp /etc/ceph/ceph.conf /etc/kolla/config/cinder/cinder-volume/

# Nova
sudo cp /etc/ceph/ceph.client.nova.keyring /etc/kolla/config/nova/
sudo cp /etc/ceph/ceph.conf /etc/kolla/config/nova/

find /etc/kolla/config -type f -name "ceph*" -exec ls -lh {} \;


In [ ]:
# ============================================================
# BƯỚC 24.7 — Verify Ceph config/keyring
# ============================================================

cat /etc/ceph/ceph.conf

sudo ceph-authtool -p /etc/ceph/ceph.client.glance.keyring || true
sudo ceph-authtool -p /etc/ceph/ceph.client.cinder.keyring || true
sudo ceph-authtool -p /etc/ceph/ceph.client.nova.keyring || true

find /etc/kolla/config -type f -name "ceph*" -print

# Nếu ceph-common có sẵn, test thử:
ceph -c /etc/ceph/ceph.conf -s || true


## BƯỚC 25 — Tạo multinode inventory

Inventory đặt tại:

```bash
/etc/kolla/multinode
```

Lưu ý:
- `[control]`: 3 Controller VM.
- `[compute]`: 3 server vật lý.
- `[storage]`: Controller VM cho service Cinder/Glance/Nova dùng Ceph backend.
- `[deployment]`: localhost.


In [ ]:
# ============================================================
# BƯỚC 25.1 — Tạo multinode inventory
# ============================================================

cat > /etc/kolla/multinode << EOF
[control]
10.0.0.11 ansible_user=ubuntu
10.0.0.12 ansible_user=ubuntu
10.0.0.13 ansible_user=ubuntu

[network]
# OVN distributed — không cần node riêng theo thiết kế lab

[compute]
# Server vật lý — bare metal compute
10.0.0.21 ansible_user=stack
10.0.0.22 ansible_user=stack
10.0.0.23 ansible_user=stack

[storage]
10.0.0.11 ansible_user=ubuntu
10.0.0.12 ansible_user=ubuntu
10.0.0.13 ansible_user=ubuntu

[monitoring]
10.0.0.11 ansible_user=ubuntu

[deployment]
localhost ansible_connection=local
EOF

cat /etc/kolla/multinode


In [ ]:
# ============================================================
# BƯỚC 25.2 — Kiểm tra ping/SSH tới toàn bộ node
# ============================================================

for ip in 10.0.0.11 10.0.0.12 10.0.0.13 10.0.0.21 10.0.0.22 10.0.0.23; do
  echo "===== PING $ip ====="
  ping -c 2 $ip
done

for ip in 10.0.0.11 10.0.0.12 10.0.0.13; do
  echo "===== SSH $ip (ubuntu) ====="
  ssh -o StrictHostKeyChecking=no ubuntu@$ip "hostname && ip -br addr | head"
done

for ip in 10.0.0.21 10.0.0.22 10.0.0.23; do
  echo "===== SSH $ip (stack) ====="
  ssh -o StrictHostKeyChecking=no stack@$ip "hostname && ip -br addr | head"
done


#mật khẩu ssh cho 11,12,13 là 123.abc
#mật khẩu ssh cho 21 là stack123

In [ ]:
# ============================================================
# BƯỚC 25.3 — Tạo SSH key cho Ansible và copy sang toàn bộ node
# ============================================================

ssh-keygen -t ed25519 -N "" -f ~/.ssh/id_ed25519

# Copy tới Controller VM (ubuntu user)
for ip in 10.0.0.11 10.0.0.12 10.0.0.13; do
  echo "===== Copy SSH key to Controller $ip (ubuntu) ====="
  ssh-copy-id -o StrictHostKeyChecking=no ubuntu@$ip
done

# Copy tới Compute bare-metal (stack user)
for ip in 10.0.0.21 10.0.0.22 10.0.0.23; do
  echo "===== Copy SSH key to Compute $ip (stack) ====="
  ssh-copy-id -o StrictHostKeyChecking=no stack@$ip
done

# Verify SSH connection
for ip in 10.0.0.11 10.0.0.12 10.0.0.13; do
  echo "===== Test SSH $ip (ubuntu) ====="
  ssh -o BatchMode=yes ubuntu@$ip "hostname"
done

for ip in 10.0.0.21 10.0.0.22 10.0.0.23; do
  echo "===== Test SSH $ip (stack) ====="
  ssh -o BatchMode=yes stack@$ip "hostname"
done


In [ ]:
# ============================================================
# BƯỚC 25.4 — Test Ansible inventory
# ============================================================

source /opt/kolla-venv/bin/activate

which ansible
ansible --version

ansible -i /etc/kolla/multinode all -m ping

# Tất cả phải SUCCESS => pong


## BƯỚC 26 — Cấu hình globals.yml

File chính:

```bash
/etc/kolla/globals.yml
```

Cần kiểm tra kỹ:
- `network_interface`
- `neutron_external_interface`
- `storage_interface`
- VIP
- Ceph MON IP


In [ ]:
# ============================================================
# BƯỚC 26.1 — Kiểm tra bridge/interface trên toàn bộ node
# ============================================================

source /opt/kolla-venv/bin/activate

ansible -i /etc/kolla/multinode all -m shell -a "ip -br addr"

ansible -i /etc/kolla/multinode all -m shell -a "ip link show br-mgmt || true"
ansible -i /etc/kolla/multinode all -m shell -a "ip link show br-storage || true"
ansible -i /etc/kolla/multinode all -m shell -a "ip link show br-ex || true"


In [ ]:
# ============================================================
# BƯỚC 26.2 — Ghi globals.yml
# ============================================================
cat > /etc/kolla/globals.yml << EOF
# ── Cơ bản ─────────────────────────────────────────────────
kolla_base_distro: ubuntu
kolla_install_type: binary
openstack_release: "2025.2"

# ── Cấu hình Registry ───────────────────────────────────────
docker_registry: "quay.io"
docker_namespace: "openstack.kolla"

# ── Network interfaces ──────────────────────────────────────
network_interface: "br-mgmt"

# Server chỉ có 1 NIC vật lý, Kolla tạo br-ex và gắn dummy0
neutron_external_interface: "dummy0"
neutron_bridge_name: "br-ex"

kolla_internal_vip_address: "10.0.0.10"
kolla_external_vip_address: "192.168.150.200"

# Storage và tunnel
storage_interface: "br-storage"
tunnel_interface: "br-mgmt"

# ── Services ────────────────────────────────────────────────
enable_openstack_core: "yes"
enable_glance: "yes"
enable_nova: "yes"
enable_neutron: "yes"
enable_cinder: "yes"
enable_horizon: "yes"
enable_heat: "no"
enable_swift: "no"

# ── Neutron OVN ─────────────────────────────────────────────
neutron_plugin_agent: "ovn"

# ── Ceph backend ────────────────────────────────────────────
glance_backend_ceph: "yes"
glance_backend_file: "no"
cinder_backend_ceph: "yes"
nova_backend_ceph: "yes"

ceph_glance_user: "glance"
ceph_glance_pool_name: "images"

ceph_cinder_user: "cinder"
ceph_cinder_pool_name: "volumes"

ceph_nova_user: "nova"
ceph_nova_pool_name: "vms"

ceph_mon_nodes: "10.0.2.21,10.0.2.22,10.0.2.23"

# FSID lấy từ: grep fsid /etc/ceph/ceph.conf
ceph_cluster_fsid: "83b388c4-46aa-11f1-8a84-000c293450cf"

# Libvirt secret UUID cho Nova + Cinder
cinder_rbd_secret_uuid: "457eb676-33da-42ec-9a8c-9293d545c337"

# ── Coordination backend ───────────────────────────────────
enable_valkey: "yes"

# ── Cinder HA / Cluster ─────────────────────────────────────
cinder_cluster_name: "cinder-cluster-1"

# ── Nova compute ────────────────────────────────────────────
nova_compute_virt_type: "kvm"

# ── Cinder backup qua Ceph ──────────────────────────────────
enable_cinder_backup: "yes"
cinder_backup_driver: "ceph"
cinder_backup_backend_ceph: "yes"
ceph_cinder_backup_user: "cinder-backup"
ceph_cinder_backup_pool_name: "backups"
EOF


cat /etc/kolla/globals.yml

(kolla-venv) ubuntu@controller-1:~$ grep -E "interface|vip|ceph|backend" /etc/kolla/globals.yml


[control]
10.0.0.11 ansible_user=ubuntu
10.0.0.12 ansible_user=ubuntu
10.0.0.13 ansible_user=ubuntu

[network]
10.0.0.11 ansible_user=ubuntu
10.0.0.12 ansible_user=ubuntu
10.0.0.13 ansible_user=ubuntu

[loadbalancer]
10.0.0.11 ansible_user=ubuntu
10.0.0.12 ansible_user=ubuntu
10.0.0.13 ansible_user=ubuntu

[compute]
10.0.0.21 ansible_user=stack
10.0.0.22 ansible_user=stack
10.0.0.23 ansible_user=stack

[storage]
10.0.0.11 ansible_user=ubuntu
10.0.0.12 ansible_user=ubuntu
10.0.0.13 ansible_user=ubuntu

[monitoring]
10.0.0.11 ansible_user=ubuntu

[deployment]
localhost ansible_connection=local

In [ ]:
# ============================================================
# BƯỚC 26.3 — Kiểm tra VIP chưa bị sử dụng
# ============================================================

# Internal VIP
ping -c 2 10.0.0.10 || true

# External VIP
ping -c 2 192.168.150.200 || true

# Nếu ping có reply rõ ràng từ host khác, cần đổi VIP.


In [ ]:
# ============================================================
# BƯỚC 26.4 — Kiểm tra Ceph MON reachability
# ============================================================

for ip in 10.0.0.21 10.0.0.22 10.0.0.23; do
  echo "===== Ceph MON check $ip ====="
  ping -c 3 $ip
done

# Nếu dùng Ceph MON trên 10.0.3.x:
for ip in 10.0.3.21 10.0.3.22 10.0.3.23; do
  echo "===== Ceph cluster check $ip ====="
  ping -c 3 $ip || true
done

ceph -c /etc/ceph/ceph.conf -s || true



# ============================================================
# BƯỚC 26.4 — Kiểm tra Ceph MON reachability
# ============================================================

# Ceph MON chạy trên storage network (10.0.2.x)
for ip in 10.0.2.21 10.0.2.22 10.0.2.23; do
  echo "===== Ceph MON check $ip ====="
  ping -c 3 $ip
done

# Đảm bảo controller có ceph client
if ! command -v ceph &> /dev/null; then
  echo "Installing ceph client..."
  sudo apt update -y
  sudo apt install -y ceph-common
fi

# Kiểm tra trạng thái cluster
echo "===== Ceph cluster status ====="
ceph -c /etc/ceph/ceph.conf -s || true


## BƯỚC 27 — Sinh mật khẩu và deploy

Các giai đoạn:
1. `kolla-genpwd`
2. đổi admin password
3. `bootstrap-servers`
4. `prechecks`
5. `pull`
6. `deploy`
7. `post-deploy`

Khuyến nghị chạy trong `tmux`.


In [ ]:
# ============================================================
# BƯỚC 27.1 — Mở tmux và activate venv
# ============================================================

tmux new -s kolla-deploy

# Trong tmux:
source /opt/kolla-venv/bin/activate
cd /etc/kolla

which kolla-ansible
kolla-ansible --version


In [ ]:
# ============================================================
# BƯỚC 27.2 — Sinh passwords
# ============================================================

cp /etc/kolla/passwords.yml /etc/kolla/passwords.yml.bak.$(date +%F-%H%M%S) || true

kolla-genpwd

ls -lh /etc/kolla/passwords.yml
grep "^keystone_admin_password:" /etc/kolla/passwords.yml


In [ ]:
# ============================================================
# BƯỚC 27.3 — Đổi admin password cho lab
# ============================================================

sed -i "s/^keystone_admin_password:.*/keystone_admin_password: 123.abc/" \
  /etc/kolla/passwords.yml

grep "^keystone_admin_password:" /etc/kolla/passwords.yml


In [ ]:
# ============================================================
# BƯỚC 27.4 — Bootstrap servers
# ============================================================

source /opt/kolla-venv/bin/activate

kolla-ansible -i /etc/kolla/multinode bootstrap-servers

# Nếu lỗi:
# kolla-ansible -i /etc/kolla/multinode bootstrap-servers -vvv


kolla-ansible bootstrap-servers -i /etc/kolla/multinode
kolla-ansible prechecks -i /etc/kolla/multinode
kolla-ansible deploy -i /etc/kolla/multinode


In [ ]:
# ============================================================
# BƯỚC 27.5 — Verify sau bootstrap
# ============================================================

ansible -i /etc/kolla/multinode all -m shell -a "docker --version || true"
ansible -i /etc/kolla/multinode all -m shell -a "systemctl is-active docker || true"
ansible -i /etc/kolla/multinode all -m shell -a "python3 --version"
ansible -i /etc/kolla/multinode all -m shell -a "free -h && df -h /"


In [ ]:
# ============================================================
# BƯỚC 27.6 — Prechecks
# ============================================================

source /opt/kolla-venv/bin/activate

kolla-ansible -i /etc/kolla/multinode prechecks

# PHẢI ĐẠT 0 FAILED
# Nếu lỗi:
# kolla-ansible -i /etc/kolla/multinode prechecks -vvv


In [ ]:
# ============================================================
# BƯỚC 27.7 — Pull Docker images
# ============================================================

source /opt/kolla-venv/bin/activate

kolla-ansible -i /etc/kolla/multinode pull

docker images | head
ansible -i /etc/kolla/multinode all -m shell -a "docker images | wc -l"


In [ ]:
# ============================================================
# BƯỚC 27.8 — Deploy OpenStack
# ============================================================

source /opt/kolla-venv/bin/activate

kolla-ansible -i /etc/kolla/multinode deploy

# Nếu lỗi:
# kolla-ansible -i /etc/kolla/multinode deploy -vvv


In [ ]:
# ============================================================
# BƯỚC 27.9 — Post-deploy
# ============================================================

source /opt/kolla-venv/bin/activate

kolla-ansible -i /etc/kolla/multinode post-deploy

ls -lh /etc/kolla/admin-openrc.sh
cat /etc/kolla/admin-openrc.sh


## BƯỚC 28 — Verify OpenStack sau deploy

In [ ]:
# ============================================================
# BƯỚC 28.1 — Cài OpenStack CLI nếu thiếu
# ============================================================

source /opt/kolla-venv/bin/activate

which openstack || pip install python-openstackclient

source /etc/kolla/admin-openrc.sh

openstack token issue
openstack service list


In [ ]:
# ============================================================
# BƯỚC 28.2 — Kiểm tra container Kolla
# ============================================================

docker ps
docker ps -a --filter "status=exited"

ansible -i /etc/kolla/multinode all -m shell -a "docker ps --format '{{.Names}}' | head -20"
ansible -i /etc/kolla/multinode all -m shell -a "docker ps -a --filter status=exited --format '{{.Names}}'"


In [ ]:
# ============================================================
# BƯỚC 28.3 — Verify OpenStack services
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

openstack service list
openstack endpoint list
openstack compute service list
openstack network agent list || true
openstack volume service list
openstack image list
openstack hypervisor list
openstack hypervisor stats show


In [ ]:
# ============================================================
# BƯỚC 28.4 — Verify Ceph backend mapping
# ============================================================

docker ps | egrep "glance|cinder|nova" || true

docker exec -it glance_api ls -lh /etc/ceph || true
docker exec -it cinder_volume ls -lh /etc/ceph || true
docker exec -it nova_compute ls -lh /etc/ceph || true

docker logs glance_api --tail 100 || true
docker logs cinder_volume --tail 100 || true
docker logs nova_compute --tail 100 || true


## Debug lỗi thường gặp

### 1. Ansible ping fail
```bash
ssh ubuntu@<ip> hostname
ssh-copy-id ubuntu@<ip>
```

### 2. Prechecks fail do interface không tồn tại
```bash
ansible -i /etc/kolla/multinode all -m shell -a "ip -br addr"
```

### 3. VIP conflict
```bash
ping 10.0.0.10
ping 192.168.150.200
```

### 4. Ceph backend lỗi
```bash
ls -lh /etc/kolla/config/glance/
ls -lh /etc/kolla/config/cinder/cinder-volume/
ls -lh /etc/kolla/config/nova/
cat /etc/ceph/ceph.conf
```

### 5. Container exited
```bash
docker ps -a --filter status=exited
docker logs <container_name> --tail 200
```

### 6. Deploy bị ngắt SSH
```bash
tmux attach -t kolla-deploy
```


## Checklist hoàn thành PHẦN VI

- [ ] SSH vào Controller VM1 được.
- [ ] `/opt/kolla-venv` hoạt động.
- [ ] `kolla-ansible --version` đúng `2025.1.x`.
- [ ] `/etc/kolla` đã có `globals.yml`, `passwords.yml`, `multinode`.
- [ ] Ceph config/keyring nằm đúng trong `/etc/kolla/config/...`.
- [ ] Inventory có 3 controller và 3 compute.
- [ ] SSH key từ Controller VM1 tới toàn bộ node đã OK.
- [ ] `ansible -i /etc/kolla/multinode all -m ping` pass.
- [ ] `globals.yml` đúng interface/VIP/Ceph backend.
- [ ] `kolla-genpwd` đã chạy.
- [ ] `keystone_admin_password` đã set cho lab.
- [ ] `bootstrap-servers` pass.
- [ ] `prechecks` pass 0 failed.
- [ ] `pull` hoàn tất.
- [ ] `deploy` hoàn tất.
- [ ] `post-deploy` tạo `/etc/kolla/admin-openrc.sh`.
- [ ] `openstack service list` chạy được.
- [ ] `openstack compute service list` thấy compute.
- [ ] `openstack volume service list` thấy cinder.
- [ ] Horizon truy cập được qua VIP/external.
